In [ ]:
# ============================================================
# Part 1 — GDP Stacked Bar Plot (IMF, by Region) 
# https://github.com/jinsungsur/data_science 
# https://jinsungsur.github.io/data_science/
# ============================================================
import pandas as pd
import plotly.graph_objects as go
import country_converter as coco

url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
import urllib.request

req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
html = urllib.request.urlopen(req).read()
tables = pd.read_html(html, match="Country/Territory")
df = tables[0]

# Flatten multi-level column headers
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ["_".join(str(c) for c in col).strip() for col in df.columns]

COUNTRY_COL = df.columns[0]
GDP_COL = [c for c in df.columns if "IMF" in c][0]

df = df[[COUNTRY_COL, GDP_COL]].copy()
df.columns = ["Country", "GDP"]
df = df[~df["Country"].str.contains(
    r"World|Africa|Asia|Europe|America|Oceania|IMF|\[", na=True)]
df["GDP"] = pd.to_numeric(
    df["GDP"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce")
df = df.dropna(subset=["GDP"])

df["Region"] = coco.convert(
    names=df["Country"].tolist(), to="UNregion", not_found="Other")

fig = go.Figure()
for region in df["Region"].unique():
    subset = df[df["Region"] == region].sort_values("GDP", ascending=False)
    fig.add_trace(go.Bar(
        name=region,
        x=[region] * len(subset),
        y=subset["GDP"],
        text=subset["Country"],
        hovertemplate="%{text}: %{y:,.0f}<extra></extra>",
    ))

fig.update_layout(
    barmode="stack",
    title="GDP by Region (IMF Forecast) — Countries Stacked",
    xaxis_title="Region",
    yaxis_title="GDP (USD Millions)",
    legend_title="Region",
)

fig.show()

with open("stacked_bar.html", "w", encoding="utf-8") as f:
    f.write(fig.to_html())

In [1]:
# ============================================================
# Part 2 — MRICloud Sankey Diagram (Type 1, ≥3 levels)
# ============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- Load multilevel lookup ---
url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
multilevel_lookup = pd.read_csv(url, sep="\t").drop("Level5", axis=1)
multilevel_lookup = multilevel_lookup.rename(columns={
    "modify": "roi",
    "modify.1": "level4",
    "modify.2": "level3",
    "modify.3": "level2",
    "modify.4": "level1",
})
multilevel_lookup = multilevel_lookup[["roi", "level4", "level3", "level2", "level1"]]

# --- Load subject data and filter to Type 1, Level 5 ---
subjectData = pd.read_csv("https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv")
subjectData = subjectData.loc[
    (subjectData.type == 1) & (subjectData.level == 5) & (subjectData.id == 127)
]
subjectData = subjectData[["roi", "volume"]]

# --- Merge with hierarchy and compute composition ---
subjectData = pd.merge(subjectData, multilevel_lookup, on="roi")
subjectData = subjectData.assign(
    icv="ICV",
    comp=subjectData.volume / np.sum(subjectData.volume),
)

# --- Define hierarchy: ICV → level1 → level2 → level3 → level4 → roi ---
level_cols = ["icv", "level1", "level2", "level3", "level4", "roi"]

all_labels = []
for col in level_cols:
    for val in subjectData[col].dropna().unique():
        label = f"{col}:{val}"
        if label not in all_labels:
            all_labels.append(label)

label_idx = {label: i for i, label in enumerate(all_labels)}

sources, targets, values = [], [], []
for src_col, tgt_col in zip(level_cols[:-1], level_cols[1:]):
    grouped = (
        subjectData
        .dropna(subset=[src_col, tgt_col])
        .groupby([src_col, tgt_col])["comp"]
        .sum()
        .reset_index()
    )
    for _, row in grouped.iterrows():
        s = label_idx[f"{src_col}:{row[src_col]}"]
        t = label_idx[f"{tgt_col}:{row[tgt_col]}"]
        sources.append(s)
        targets.append(t)
        values.append(row["comp"])

# --- Strip prefix for display labels ---
display_labels = [lbl.split(":", 1)[1] for lbl in all_labels]

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15, thickness=20,
        line=dict(color="black", width=0.5),
        label=display_labels,
    ),
    link=dict(source=sources, target=targets, value=values),
))

fig.update_layout(
    title_text="MRICloud Brain Volume (Type 1) — Sankey Diagram",
    font_size=11,
)

fig.show()

with open("sankey.html", "w", encoding="utf-8") as f:
    f.write(fig.to_html())